In [ ]:
%matplotlib inline

# Training a Classifier

When working with image, text, audio, or video data, standard Python libraries are commonly used to load the data into NumPy arrays, which can then be converted into `torch.*Tensor` objects.

* For image processing, libraries such as Pillow and OpenCV are widely used.
* For audio processing, packages like SciPy and librosa are commonly employed.
* For text processing, options include native Python or Cython-based loaders, as well as libraries such as NLTK and SpaCy.

For computer vision tasks specifically, PyTorch provides the `torchvision` package, which includes utilities for loading popular datasets such as ImageNet, CIFAR-10, and MNIST, along with image transformation tools through `torchvision.datasets` and `torch.utils.data.DataLoader`.

These utilities significantly simplify the workflow by reducing the need for repetitive boilerplate code.

In this tutorial, we will work with the CIFAR-10 dataset, which contains the following classes:

`airplane`, `automobile`, `bird`, `cat`, `deer`, `dog`, `frog`, `horse`, `ship`, and `truck`.

Each image in CIFAR-10 is a 32×32 RGB image, meaning it has three color channels and dimensions of 32 by 32 pixels.

## Training an Image Classifier

The tutorial will proceed through the following steps:

1. Load and normalize the CIFAR-10 training and test datasets using `torchvision`
2. Define a Convolutional Neural Network (CNN)
3. Specify a loss function
4. Train the network using the training dataset
5. Evaluate the trained model on the test dataset

### 1. Loading and Normalizing CIFAR-10

Using `torchvision`, loading the CIFAR-10 dataset is straightforward and requires only a few lines of code.



In [ ]:
import torch
import torchvision
from torchvision.transforms import v2

The output of torchvision datasets are PILImage images of range \[0,
1\]. We transform them to Tensors of normalized range \[-1, 1\].


In [ ]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

Let's explore some of the training images.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# functions to show an image


def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


# get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# show images
imshow(torchvision.utils.make_grid(images))
# print labels
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(batch_size)))

# 2. Define a Convolutional Neural Network

Below, we will build a Simple Convolutional Neural Network (CNN) in PyTorch.

This module defines a basic CNN architecture for image classification tasks.
The network consists of:
    - Two convolutional layers
    - ReLU activation functions
    - Max-pooling layers
    - Three fully connected (dense) layers

The architecture is suitable for small image datasets such as CIFAR-10.

Classes:
    Net: A convolutional neural network model.

Example:
    >>> net = Net()
    >>> print(net)

    

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):
    """
    Convolutional Neural Network for image classification.
    Architecture:
        Input -> Conv1 -> ReLU -> MaxPool
              -> Conv2 -> ReLU -> MaxPool
              -> Flatten
              -> FC1 -> ReLU
              -> FC2 -> ReLU
              -> FC3 -> Output

    Input Shape:
        (batch_size, 3, 32, 32)
    Output Shape:
        (batch_size, 10)
    Attributes:
        conv1 (nn.Conv2d):
            First convolutional layer with:
                - 3 input channels (RGB image)
                - 6 output channels
                - 5x5 kernel size

        pool (nn.MaxPool2d):
            Max pooling layer with:
                - 2x2 window
                - stride of 2

        conv2 (nn.Conv2d):
            Second convolutional layer with:
                - 6 input channels
                - 16 output channels
                - 5x5 kernel size

        fc1 (nn.Linear):
            First fully connected layer mapping
            flattened features to 120 neurons.

        fc2 (nn.Linear):
            Second fully connected layer mapping
            120 features to 84 neurons.

        fc3 (nn.Linear):
            Final output layer mapping
            84 features to 10 output classes.
    """

    def __init__(self): 
        """Initialize the neural network layers."""
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        """
        Perform a forward pass through the network.

        Args:
            x (torch.Tensor):
                Input tensor of shape
                (batch_size, 3, 32, 32).

        Returns:
            torch.Tensor:
                Output logits for each class
                with shape (batch_size, 10).
        """
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()

# 3. Define a Loss function and optimizer

For multiclass classification, we will use a Classification Cross-Entropy loss and SGD with momentum.


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

# 4. Train the network


This code below implements the training loop for a neural network in PyTorch.

The outer loop runs for two epochs, meaning the model processes the entire training dataset twice. During each epoch, the code iterates through batches of training data provided by `trainloader`.

For each mini-batch:

1. The input images and their corresponding labels are extracted.
2. The gradients stored in the optimizer are reset using `optimizer.zero_grad()`. This is necessary because PyTorch accumulates gradients by default.
3. The input data is passed through the neural network using `net(inputs)` to produce predictions.
4. The loss between the predictions and the true labels is computed using the loss function `criterion`.
5. Backpropagation is performed with `loss.backward()`, which calculates the gradients of the loss with respect to the network parameters.
6. The optimizer updates the model parameters using `optimizer.step()`.

The variable `running_loss` keeps track of the accumulated training loss. Every 2000 mini-batches, the average loss is printed to monitor training progress, and the running loss counter is reset.

After all epochs are completed, the message `"Finished Training"` is printed, indicating that the model training process has ended.



In [ ]:
for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')

Let\'s save our trained model:


In [ ]:
PATH = './cifar_net.pt'
torch.save(net.state_dict(), PATH)

See [here](https://pytorch.org/docs/stable/notes/serialization.html) for
more details on saving PyTorch models.

# 5. Test the network on the test data

We have trained the network for 2 epochs over the training dataset. But we need to check if the network has learnt anything at all.

We will check this by predicting the class label that the neural network outputs, and checking it against the ground-truth. If the prediction is correct, we add the sample to the list of correct predictions.

First step is to display an image from the test set to get familiar.


In [ ]:
dataiter = iter(testloader)
images, labels = next(dataiter)

# print images
imshow(torchvision.utils.make_grid(images))
print('GroundTruth: ', ' '.join(f'{classes[labels[j]]:5s}' for j in range(4)))

Next, let\'s load our saved model (note: saving and re-loading the model wasn\'t necessary here, we only did it to illustrate how to do so):

In [ ]:
net = Net()
net.load_state_dict(torch.load(PATH, weights_only=True))

Okay, now let us see what the neural network thinks these examples above
are:


In [ ]:
outputs = net(images)

The outputs are energies for the 10 classes. The higher the energy for a
class, the more the network thinks that the image is of the particular
class. So, let\'s get the index of the highest energy:


In [ ]:
_, predicted = torch.max(outputs, 1)

print('Predicted: ', ' '.join(f'{classes[predicted[j]]:5s}'
                              for j in range(4)))

The results seem pretty good.

Let us look at how the network performs on the whole dataset.


In [ ]:
correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in testloader:
        images, labels = data
        # calculate outputs by running images through the network
        outputs = net(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

The model performs significantly better than random guessing, which would yield an accuracy of only 10% for a 10-class classification problem. This suggests that the network has successfully learned meaningful patterns from the data.

Let's to examine which classes the model predicts accurately and which classes it struggles with.

In [ ]:
# prepare to count predictions for each class
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

# again no gradients needed
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predictions = torch.max(outputs, 1)
        # collect the correct predictions for each class
        for label, prediction in zip(labels, predictions):
            if label == prediction:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1


# print accuracy for each class
for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')